# 05 — MLX Conversion for Apple Silicon Inference

**What this notebook does and why it exists**

After training on Colab, we want to run the model locally on an M1 MacBook Air. This notebook converts the fine-tuned model into the MLX format, which is optimised for Apple Silicon's unified memory architecture.

The conversion process has three stages:
1. Merge the LoRA adapter into the base model weights (producing a single, standalone Hugging Face model)
2. Convert the merged model to MLX format with 4-bit quantisation
3. Verify the conversion by running a test inference

The result is a zip archive on your Google Drive that you can download and use immediately on your Mac.

**Prerequisites:** Run `03_rl_grpo_training.ipynb` first. This notebook expects the final GRPO adapter to be in `podcast_translation/grpo_adapter/final/` on your Drive.

---
## What MLX is and why it is the right choice for Apple Silicon

MLX is a numerical computation framework developed by Apple's machine learning research team, designed specifically for Apple Silicon chips (M1, M2, M3, M4 series). It differs from alternatives in a fundamental way:

- **Unified memory:** Apple Silicon has a single memory pool shared between the CPU and GPU (and the Neural Engine). MLX is designed to exploit this — tensors do not need to be copied between devices, so even a model that doesn't fit on the "GPU" can run efficiently because there is no GPU-CPU boundary.
- **llama.cpp** is another popular option. It is more mature, supports a wider range of quantisation formats (GGUF), and runs on almost any hardware. However, it is not optimised for MLX's lazy evaluation and operator fusion, so on Apple Silicon it is often 15–30% slower than mlx-lm for the same model.
- **Ollama** is a user-friendly wrapper around llama.cpp. The same trade-offs apply: easy to use, but not Apple Silicon-native at the level that MLX is.

For a 1.5B model on an M1 MacBook Air with 8 GB unified memory, MLX with 4-bit quantisation is the recommended approach. The model will use approximately 1 GB of memory, leaving ample room for the tokeniser, KV cache, and the macOS system.

**Known compatibility:** Qwen2.5 is well-supported by mlx-lm. The mlx-community on Hugging Face has published multiple pre-converted Qwen2.5 models at various sizes, confirming that the architecture is stable in MLX.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q \
    transformers==4.45.0 \
    peft==0.13.2 \
    torch==2.4.0 \
    mlx-lm==0.19.3

# Note: mlx-lm v0.19.3 is stable for Qwen2.5 conversion.
# If you see version conflicts, the convert command syntax remains the same
# across recent versions: --hf-path, --mlx-path, -q, --q-bits

print('Dependencies installed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import date

BASE_DIR     = '/content/drive/MyDrive/podcast_translation'
GRPO_ADAPTER = os.path.join(BASE_DIR, 'grpo_adapter', 'final')
MERGED_DIR   = os.path.join(BASE_DIR, 'merged_model')
MLX_DIR      = os.path.join(BASE_DIR, 'mlx_model')

TODAY        = date.today().strftime('%Y-%m-%d')
ARCHIVE_NAME = f'podcast_translation_mlx_{TODAY}.zip'
ARCHIVE_PATH = os.path.join(BASE_DIR, ARCHIVE_NAME)

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(MLX_DIR, exist_ok=True)

BASE_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Merged model will go to: {MERGED_DIR}')
print(f'MLX model will go to:    {MLX_DIR}')
print(f'Archive name:            {ARCHIVE_NAME}')

---
## Decision note: Why merge the LoRA adapter before MLX conversion?

LoRA adapters work by storing *delta matrices* — small low-rank updates that are added to the frozen base model weights at inference time. A LoRA adapter is not a standalone model; it requires the base model to function.

mlx-lm's `convert` tool converts a *complete* Hugging Face model (all weights in one directory). It does not understand the PEFT adapter format. Therefore, we must first merge the adapter deltas back into the base weights, producing a single standard model that mlx-lm can consume.

**What is lost by merging?**
- Once merged, you cannot cheaply swap or stack adapters. If you wanted to have a "formal English" adapter and a "casual English" adapter that you could switch at runtime, you would need to keep them separate.
- You cannot easily compare the merged model against the base model without re-downloading the base weights.
- The merged model is approximately the same size as the base model (~3 GB in bfloat16), whereas a LoRA adapter is only ~50–100 MB.

**What is gained by merging?** A portable, self-contained model that can be consumed by any inference framework — MLX, llama.cpp, vLLM — without needing a PEFT library.

In [ ]:
# ── Merge LoRA adapter into base model ───────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print(f'Loading base model: {BASE_MODEL_NAME}')
tokeniser = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

print(f'Loading GRPO LoRA adapter from: {GRPO_ADAPTER}')
merged_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER)

print('Merging adapter into base weights...')
merged_model = merged_model.merge_and_unload()
print('Merge complete.')

# Verify the merged model has no PEFT wrappers
print(f'Model type: {type(merged_model).__name__}')
print(f'Total parameters: {sum(p.numel() for p in merged_model.parameters()):,}')

In [ ]:
# ── Save merged model to Google Drive ────────────────────────────────────────
print(f'Saving merged model to: {MERGED_DIR}')
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokeniser.save_pretrained(MERGED_DIR)

# Verify all necessary files are present before conversion
required_files = ['config.json', 'tokenizer.json', 'tokenizer_config.json']
saved_files    = os.listdir(MERGED_DIR)
print(f'\nFiles in merged model directory: {sorted(saved_files)}')

for req in required_files:
    status = '✓' if req in saved_files else '✗ MISSING'
    print(f'  {req}: {status}')

safetensors_files = [f for f in saved_files if f.endswith('.safetensors')]
print(f'  Safetensors shards: {len(safetensors_files)}')

if not safetensors_files:
    raise RuntimeError('No safetensors files found. Conversion will fail without model weights.')

print('\nMerged model verified. Ready for MLX conversion.')

---
## Decision note: 4-bit vs 8-bit quantisation for a 1.5B model on Apple Silicon

| | **4-bit (q4)** | **8-bit (q8)** |
|---|---|---|
| **Memory (1.5B model)** | ~1.0 GB | ~1.8 GB |
| **Speed** | Faster (less memory bandwidth) | Slightly slower |
| **Quality** | Small quality loss (~0.5–1 COMET point) | Near-identical to bfloat16 |
| **Recommended for 8 GB Mac** | ✅ Yes | ✅ Yes (both fit easily) |
| **Recommended for 16 GB Mac** | ✅ Yes | ✅ Prefer if quality matters |

For a 1.5B model, both options are viable even on an 8 GB Mac. We default to **4-bit** because:
1. The memory footprint is small enough that speed is the bottleneck, and 4-bit is faster
2. The quality loss at 1.5B is measurable (~0.5 COMET points) but not human-perceptible in most cases
3. It leaves more memory for the operating system and other applications

**To use 8-bit instead:** Change `--q-bits 4` to `--q-bits 8` in the convert command below.

**What happens to the model during quantisation?** Each bfloat16 weight (2 bytes) is represented by a 4-bit integer (0.5 bytes) plus a small per-group scaling factor. Weights within each group are mapped to the nearest of 16 possible values. This introduces rounding error, which accumulates through the network — which is why the quality loss is small but not zero.

In [ ]:
# ── Convert to MLX format ─────────────────────────────────────────────────────
# Command syntax (verified with mlx-lm 0.19.3):
#   mlx_lm.convert --hf-path <source> --mlx-path <dest> -q --q-bits <4 or 8>
#
# --hf-path : path to the merged HuggingFace model directory (can be local or Hub)
# --mlx-path: output directory for the converted MLX model
# -q        : enable quantisation (without this, saves as bfloat16, ~3 GB)
# --q-bits  : quantisation bit width (4 or 8; default when -q is used: 4)

QUANTISATION_BITS = 4   # Change to 8 for higher quality (see Decision note above)

convert_cmd = (
    f'mlx_lm.convert '
    f'--hf-path "{MERGED_DIR}" '
    f'--mlx-path "{MLX_DIR}" '
    f'-q --q-bits {QUANTISATION_BITS}'
)

print(f'Running: {convert_cmd}')
!{convert_cmd}

# Verify conversion output
mlx_files = os.listdir(MLX_DIR)
print(f'\nMLX model files: {sorted(mlx_files)}')

# Expected: config.json, model.safetensors (or sharded), tokenizer files
if not any(f.endswith('.safetensors') for f in mlx_files):
    print('WARNING: No safetensors found in MLX directory. Conversion may have failed.')
else:
    print('Conversion appears successful.')

In [ ]:
# ── Verify conversion with test inference ─────────────────────────────────────
# We run mlx_lm.generate as a subprocess to verify the model works.
# Note: mlx requires actual Apple Silicon to run. On Colab (x86/TPU),
# this will import but may raise an error at model load time.
# If it fails here, the zip archive is still valid — test on your Mac.

import subprocess

TEST_PROMPT = (
    'Translate the following German podcast transcript to natural English.\n\n'
    'German: Das ist ein wichtiger Punkt, den wir alle berücksichtigen sollten.\n\n'
    'Translation:'
)

# On Colab (non-Apple hardware), mlx falls back to CPU simulation which is slow
# but functional for a quick syntax/format check.
verify_cmd = [
    'python', '-c',
    f"""
try:
    from mlx_lm import load, generate
    model, tokenizer = load('{MLX_DIR}')
    result = generate(model, tokenizer,
                      prompt='{TEST_PROMPT.replace(chr(10), ' ')}',
                      max_tokens=60, verbose=False)
    print('TRANSLATION:', result)
    print('MLX inference PASSED')
except Exception as e:
    print(f'MLX inference note: {{e}}')
    print('If running on Colab, this is expected. Test on Apple Silicon Mac.')
"""
]

result = subprocess.run(verify_cmd, capture_output=True, text=True, timeout=120)
print(result.stdout)
if result.returncode != 0:
    print('stderr:', result.stderr[:500])

In [ ]:
# ── Package as zip archive ────────────────────────────────────────────────────
import zipfile
from tqdm.auto import tqdm

print(f'Creating archive: {ARCHIVE_PATH}')
mlx_files_list = []
for root, dirs, files in os.walk(MLX_DIR):
    for f in files:
        mlx_files_list.append(os.path.join(root, f))

with zipfile.ZipFile(ARCHIVE_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in tqdm(mlx_files_list, desc='Archiving'):
        arcname = os.path.relpath(file_path, BASE_DIR)
        zf.write(file_path, arcname)

archive_size_mb = os.path.getsize(ARCHIVE_PATH) / (1024 ** 2)
print(f'Archive created: {ARCHIVE_NAME} ({archive_size_mb:.0f} MB)')
print(f'Download it from: {ARCHIVE_PATH}')
print('\nTo find it in Drive: My Drive → podcast_translation → ' + ARCHIVE_NAME)

---
## What to do on your Mac after downloading

### Step 1: Install mlx-lm

```bash
pip install mlx-lm
```

Or, to install in a new virtual environment (recommended to avoid conflicts):

```bash
python3 -m venv mlx-env
source mlx-env/bin/activate
pip install mlx-lm
```

### Step 2: Unzip the archive

```bash
cd ~/Downloads
unzip podcast_translation_mlx_YYYY-MM-DD.zip
# This creates a directory: mlx_model/
```

### Step 3: Run a translation from the terminal

```bash
mlx_lm.generate \
  --model ~/Downloads/mlx_model \
  --prompt "Translate the following German podcast transcript to natural English.\n\nGerman: Das finde ich wirklich interessant.\n\nTranslation:"
```

### Step 4: Run the inference notebook locally

Open `06_inference.ipynb` in Jupyter locally (not Colab). Point the `MODEL_PATH` variable to your unzipped `mlx_model/` directory. The notebook handles chunking and batch translation.

### Known caveats

- **First-run compilation:** On the first call to `mlx_lm.generate`, Metal shaders are compiled for your specific chip. This takes 15–60 seconds and produces a compilation message. Subsequent calls are fast.
- **Memory pressure:** Ensure you have at least 2 GB of free unified memory. Close memory-intensive applications (e.g., Chrome with many tabs) before running long translation jobs.
- **Model behaviour vs Colab:** See the note below on expected differences after quantisation.

---
## What happens to model behaviour after quantisation

The MLX 4-bit model is *not* identical to the Colab bfloat16 model. Here is what to expect:

**Differences that are real but small:**
- Individual word choices may differ on some sentences — the model has slightly different weights, so it assigns slightly different probabilities to each token
- Very long translations (> 80 tokens) may diverge more significantly, because rounding errors accumulate over the generation
- Rare German words that required precise weight values may be translated differently

**Differences that should be negligible in practice:**
- Overall translation quality (COMET) degrades by approximately 0.5–1 points at 4-bit quantisation for a 1.5B model — this is below the threshold of human perception in side-by-side comparison
- Adequacy (preserving meaning) is barely affected; fluency (naturalness) degrades very slightly

**How to test:** Translate 20–30 sentences using both the Colab model (via `06_inference.ipynb` on Colab) and the MLX model on your Mac. If you see systematic differences (e.g., the Mac model always produces shorter outputs, or uses more formal register), try 8-bit quantisation instead.

**Bottom line:** For a podcast translation use case, the 4-bit MLX model should be indistinguishable from the Colab model in 9 out of 10 sentences.